In [2]:
# standardní knihovny Pythonu
import sys
import re
import time
from pathlib import Path
from urllib.parse import urljoin

# knihovny třetích stran
import pandas as pd
import requests
from bs4 import BeautifulSoup

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené konstanty a společné schéma datasetu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS

# základní technické čištění textu
from utils.text_cleaning import (
    clean_base_text,
    join_text_parts
)

In [3]:
# akademický rok sbíraných dat
YEAR = 2026

# tematické oblasti použité pro vyhledávání předmětů
SUBJECTS = [
    "Accounting",
    "Economics",
    "Finance",
    "Information Systems",
    "Data Science",
    "Computer Science",
    "Statistics",
    "Mathematics",
    "Management",
    "Marketing",
]

# URL adresy zdrojového katalogu
BASE_LIST_URL = "https://courseinfo.canterbury.ac.nz/GetCourses.aspx"
BASE_DOMAIN = "https://courseinfo.canterbury.ac.nz/"

In [4]:
# určení úrovně předmětu podle číselné části kódu
def get_course_level(course_code):

    match = re.search(r"[A-Z]{4}(\d{3})", course_code)

    if not match:
        return None

    return int(match.group(1)[0])


# kontrola undergraduate úrovně předmětu
# University of Canterbury klasifikuje úrovně 100–300 jako undergraduate
def is_undergraduate(course_code):

    level = get_course_level(course_code)

    return level in [1, 2, 3]


# převod označení semestru do sjednocené podoby datasetu
def normalize_uc_semester(semester_code):

    if semester_code == "S1":
        return "summer"

    if semester_code == "S2":
        return "winter"

    return None


# kontrola standardní on-campus varianty výuky
def is_standard_occurrence(occurrence):

    return (
        ("S1" in occurrence or "S2" in occurrence)
        and "(C)" in occurrence
    )


# určení priority occurrence při výběru varianty předmětu
def occurrence_priority(occurrence):

    if "S1" in occurrence and "(C)" in occurrence:
        return 1

    if "S2" in occurrence and "(C)" in occurrence:
        return 2

    return 99

In [5]:
# získání odkazů na detailní stránky předmětů pro zvolenou tematickou oblast
def collect_subject_links(subject_name):

    url = f"{BASE_LIST_URL}?subjectnames={subject_name.replace(' ', '+')}&year={YEAR}"

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()
    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(response.text, "html.parser")

    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "GetCourseDetails.aspx" not in href:
            continue

        full_url = urljoin(BASE_DOMAIN, href)

        course_match = re.search(r"course=([A-Z]{4}\d{3})", href)
        occurrence_match = re.search(r"occurrence=([^&]+)", href)

        if not course_match or not occurrence_match:
            continue

        course_code = course_match.group(1)
        occurrence = occurrence_match.group(1)

        if not is_undergraduate(course_code):
            continue

        if not is_standard_occurrence(occurrence):
            continue

        links.append({
            "subject": subject_name,
            "course_code": course_code,
            "occurrence": occurrence,
            "detail_url": full_url,
            "priority": occurrence_priority(occurrence),
            "source_list_url": url,
        })

    return links

In [6]:
# získání odkazů na detailní stránky předmětů
all_links = []

for subject in SUBJECTS:

    print(f"Stahuji seznam: {subject}")

    subject_links = collect_subject_links(subject)

    print(f"  nalezeno vhodných occurrences: {len(subject_links)}")

    all_links.extend(subject_links)

    time.sleep(1)

links_df = pd.DataFrame(all_links)

print("Celkem nalezených occurrences:", len(links_df))
print("Počet unikátních course codes:")
print(links_df["course_code"].nunique())

Stahuji seznam: Accounting
  nalezeno vhodných occurrences: 22
Stahuji seznam: Economics
  nalezeno vhodných occurrences: 31
Stahuji seznam: Finance
  nalezeno vhodných occurrences: 20
Stahuji seznam: Information Systems
  nalezeno vhodných occurrences: 18
Stahuji seznam: Data Science
  nalezeno vhodných occurrences: 6
Stahuji seznam: Computer Science
  nalezeno vhodných occurrences: 25
Stahuji seznam: Statistics
  nalezeno vhodných occurrences: 25
Stahuji seznam: Mathematics
  nalezeno vhodných occurrences: 29
Stahuji seznam: Management
  nalezeno vhodných occurrences: 40
Stahuji seznam: Marketing
  nalezeno vhodných occurrences: 20
Celkem nalezených occurrences: 236
Počet unikátních course codes:
192


In [7]:
# výběr jedné occurrence pro každý předmět
links_selected_df = (
    links_df
    .sort_values(["course_code", "priority"])
    .drop_duplicates(subset=["course_code"], keep="first")
    .reset_index(drop=True)
)

print("Počet unikátních kurzů:", len(links_selected_df))
print("Počet podle subject:")
print(links_selected_df["subject"].value_counts())

Počet unikátních kurzů: 192
Počet podle subject:
subject
Management             37
Economics              28
Mathematics            26
Computer Science       22
Accounting             19
Statistics             17
Information Systems    16
Marketing              12
Finance                 9
Data Science            6
Name: count, dtype: int64


In [8]:
# extrakce textového obsahu podle HTML id elementu
def extract_text_by_id(soup, element_id):

    element = soup.find(id=element_id)

    if not element:
        return pd.NA

    return clean_base_text(
        element.get_text(" ", strip=True)
    )


# extrakce způsobu hodnocení předmětu
def extract_assessment(soup):

    rows = soup.select("#ctl00_ContentPlaceHolder1_AssessmentsDiv tr")

    assessments = []

    for row in rows:
        cols = row.find_all("td")

        if len(cols) < 3:
            continue

        assessment_type = clean_base_text(
            cols[0].get_text(" ", strip=True)
        )

        percentage = clean_base_text(
            cols[2].get_text(" ", strip=True)
        )

        if pd.isna(assessment_type):
            continue

        if not pd.isna(percentage):
            assessments.append(f"{assessment_type} ({percentage})")

        else:
            assessments.append(assessment_type)

    return join_text_parts(assessments)


# extrakce literatury předmětu
def extract_literature(soup):

    textbook_div = soup.find(id="ctl00_ContentPlaceHolder1_TextbookDiv")

    if not textbook_div:
        return pd.NA

    return clean_base_text(
        textbook_div.get_text(" ", strip=True)
    )

In [9]:
# extrakce obsahu meta tagu podle názvu
def get_meta_content(soup, name):

    tag = soup.find("meta", attrs={"name": name})

    if not tag:
        return pd.NA

    return clean_base_text(
        tag.get("content")
    )


# parsování detailní stránky předmětu
def parse_uc_detail_page(detail_url):

    response = requests.get(
        detail_url,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()
    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(response.text, "html.parser")

    credits_raw = get_meta_content(soup, "UC.coursepoints")
    credits = float(credits_raw) if not pd.isna(credits_raw) else pd.NA

    semester_code = get_meta_content(soup, "UC.semestercode")

    return {
        "course_title": get_meta_content(soup, "UC.coursetitle"),
        "course_code": get_meta_content(soup, "UC.course"),
        "university": "University of Canterbury",
        "country": "New Zealand",
        "source_url": detail_url,
        "academic_year": get_meta_content(soup, "UC.courseyear"),
        "semester": get_meta_content(soup, "UC.semester"),
        "semester_normalized": normalize_uc_semester(semester_code),
        "study_level": "undergraduate",
        "credits": credits,
        "credit_system": "UC points",

        # orientační převod podle doporučení OZS:
        # 2 local credits = 1 ECTS
        "ects_credits": credits / 2 if not pd.isna(credits) else pd.NA,

        "aims": extract_text_by_id(
            soup,
            "ctl00_ContentPlaceHolder1_CourseOccurrenceDescriptionLabel"
        ),

        "course_content": pd.NA,

        "learning_outcomes": extract_text_by_id(
            soup,
            "ctl00_ContentPlaceHolder1_LearningOutcomesLabel"
        ),

        "assessment_methods": extract_assessment(soup),

        "literature": extract_literature(soup),
    }

In [10]:
# stažení a parsování detailních stránek předmětů
results = []
errors = []

for idx, row in links_selected_df.iterrows():

    course_code = row["course_code"]
    detail_url = row["detail_url"]

    print(f"{idx + 1}/{len(links_selected_df)} | {course_code}")

    try:
        record = parse_uc_detail_page(detail_url)
        results.append(record)

    except Exception as e:
        errors.append({
            "course_code": course_code,
            "detail_url": detail_url,
            "error": str(e)
        })

    time.sleep(1)

df = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

print("\nHotovo.")
print("Stažené předměty:", len(df))
print("Chyby:", len(errors_df))

1/192 | ACCT102
2/192 | ACCT103
3/192 | ACCT156
4/192 | ACCT211
5/192 | ACCT222
6/192 | ACCT252
7/192 | ACCT254
8/192 | ACCT256
9/192 | ACCT312
10/192 | ACCT316
11/192 | ACCT332
12/192 | ACCT340
13/192 | ACCT341
14/192 | ACCT346
15/192 | ACCT347
16/192 | ACCT356
17/192 | ACCT358
18/192 | ACCT359
19/192 | ACCT390
20/192 | BSNS390
21/192 | COSC101
22/192 | COSC121
23/192 | COSC122
24/192 | COSC131
25/192 | COSC260
26/192 | COSC261
27/192 | COSC262
28/192 | COSC264
29/192 | COSC265
30/192 | COSC345
31/192 | COSC362
32/192 | COSC363
33/192 | COSC364
34/192 | COSC367
35/192 | COSC368
36/192 | COSC369
37/192 | DATA101
38/192 | DATA201
39/192 | DATA203
40/192 | DATA301
41/192 | DATA303
42/192 | DATA309
43/192 | DIGI103
44/192 | ECON104
45/192 | ECON105
46/192 | ECON206
47/192 | ECON207
48/192 | ECON208
49/192 | ECON213
50/192 | ECON222
51/192 | ECON225
52/192 | ECON314
53/192 | ECON321
54/192 | ECON323
55/192 | ECON324
56/192 | ECON325
57/192 | ECON326
58/192 | ECON329
59/192 | ECON331
60/192

In [12]:
# seřazení sloupců podle společného schématu datasetu
df = df[FINAL_COLUMNS]

print("Počet předmětů:", len(df))

Počet předmětů: 192


In [13]:
# export naparsovaných sylabů University of Canterbury do interim vrstvy
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "canterbury_syllabi_raw_parsed.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Uloženo {len(df)} kurzů do {OUTPUT_PATH.name}")

Uloženo 192 kurzů do canterbury_syllabi_raw_parsed.csv
